In [1037]:
import pandas as pd
import os
import numpy as np

In [1038]:

data_path = r'C:\Users\fadia\OneDrive\Desktop\Ody\Scripts\Transformed Data\Master Combined\Master_Sales_Data_2021-08-19_to_2025-07-22.csv'
df=pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])

In [1039]:
original_happy_hour_path_csv =r"C:\Users\fadia\OneDrive\Desktop\Ody\Data\original_daily_happy_hour.csv"
original_happy_hour_path_excel =r"C:\Users\fadia\OneDrive\Desktop\Ody\Data\daily_happy_hour.xlsx"
# Load the original happy hour data
# df_happy_hour = pd.read_csv(original_happy_hour_path_csv)
# df_happy_hour["Date"] = pd.to_datetime(df_happy_hour["Date"])
# df_happy_hour = df_happy_hour.sort_values(by=["Date"])
df_happy_hour_excel = pd.read_excel(original_happy_hour_path_excel)
df_happy_hour_excel["Date"] = pd.to_datetime(df_happy_hour_excel["Date"])
df_happy_hour_excel = df_happy_hour_excel.sort_values(by=["Date"])
df_happy_hour = df_happy_hour_excel
df_happy_hour

,Date,Happy_Hour,Discount
0,2021-08-19,Happy Hour 12-6,40
1,2021-08-20,Happy Hour 12-6,40
2,2021-08-21,Happy Hour 12-6,40
3,2021-08-22,Happy Hour 12-6,40
4,2021-08-23,Happy Hour 12-6,40
...,...,...,...
1429,2025-07-18,Happy Hour 12-7,40
1430,2025-07-19,Happy Hour 12-7,40
1431,2025-07-20,Happy Hour 12-7,40
1432,2025-07-21,Happy Hour 12-7,40


In [1040]:
# Load beverage items excluding bottles by glass
items_file_path = r"C:\Users\fadia\OneDrive\Desktop\Ody\Data\beverage_items_excluding_btl_by_gls.xlsx"
df_items = pd.read_excel(items_file_path)

In [1041]:
# Keep only items that are in the df_items DataFrame
df= df[df['Item'].isin(df_items['Item'])]


In [1042]:
# get the unique items from the items dataframe
unique_items = df_items['Item'].unique()
unique_items_df = pd.DataFrame(unique_items, columns=['Item'])
unique_items_df.sort_values(by=['Item'], inplace=True)
unique_items_df.reset_index(drop=True, inplace=True)
unique_items_df

,Item
0,1800 Reposado- Gls
1,1800 Silver - Gls
2,A.Fuente Conquistadores
3,A.Fuente Gran
4,Aberfeldy 16Y. Gls
...,...
313,Totino Cherry - Gls
314,Totino Peach - Gls
315,West Cork - Gls
316,White Russian


In [1043]:
# Create output folder if it doesn't exist
output_folder = r"C:\Users\fadia\OneDrive\Desktop\Ody\Scripts\Beverage by Glass Price History"
# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [1052]:
# Example usage
# item_name = "Amstel Pint"
# item_name = "Russian Standard Gls"
# item_name = "Almaza"
# item_name= "Almond"
# item_name ="Corona"
# item_name= "Budweiser"
# item_name ="Beer Mahou"
# item_name= "Carakale Blue Valley"
# item_name="Carlsberg"
# item_name = "Espresso"
# item_name ="G Bitter Lemon"
# item_name="Heineken"
# item_name ="Herbal Tea"
# item_name ="Sol Bucket 5"
# item_name="Solan Water"
# item_name="Sparkling Water S"
item_name="Stella Pitcher"
# item_name = "J.W Black Label Gls"
# Filter df for the specific item
item_df = df[df['Item'] == item_name].copy()
# get start_date and end_date for the item dataframe
if item_df.empty:
    print(f"No data found for item: {item_name}")
    exit()
if not item_df.empty:
    start_date = item_df['Date'].min().normalize()
    end_date = item_df['Date'].max().normalize()


In [1053]:
item_df = item_df[['Date', 'Price','Quantity','Total_Price','Table','Payment']]
item_df.reset_index(drop=True, inplace=True)
item_df

,Date,Price,Quantity,Total_Price,Table,Payment
0,2021-09-09 21:15:59,24.0,1,24.0,45,Cash
1,2021-09-09 22:53:35,24.0,1,24.0,45,Cash
2,2021-10-21 21:19:05,24.0,1,24.0,4,Visa Card
3,2021-11-19 14:36:54,14.4,1,14.4,7,Visa Card
4,2022-04-06 15:17:36,14.4,1,14.4,25,Cash
5,2022-04-06 16:10:41,14.4,1,14.4,25,Cash
6,2022-04-09 16:32:33,14.4,1,14.4,24,Visa Card
7,2022-04-13 16:03:33,14.4,1,14.4,24,Visa Card
8,2022-04-13 17:00:58,14.4,1,14.4,24,Visa Card
9,2022-04-13 17:57:40,14.4,1,14.4,24,Visa Card


In [1054]:
# drop duplicates , keep the first occurrence
item_df = item_df.drop_duplicates(keep='first')


In [1055]:
# Keep only Date and Price columns
item_df = item_df[['Date', 'Price']]
# Reset index
item_df.reset_index(drop=True, inplace=True)
item_df

,Date,Price
0,2021-09-09 21:15:59,24.0
1,2021-09-09 22:53:35,24.0
2,2021-10-21 21:19:05,24.0
3,2021-11-19 14:36:54,14.4
4,2022-04-06 15:17:36,14.4
5,2022-04-06 16:10:41,14.4
6,2022-04-09 16:32:33,14.4
7,2022-04-13 16:03:33,14.4
8,2022-04-13 17:00:58,14.4
9,2022-04-13 17:57:40,14.4


In [1056]:
# Vectorized version of assign_time_slot function
def assign_time_slot_vectorized(hour_series: pd.Series) -> pd.Series:
    # Define conditions and choices
    conditions = [
        (hour_series >= 12) & (hour_series < 18), # 12:00 to 18:00
        (hour_series >= 18) & (hour_series < 19), # 18:00 to 19:00
        ((hour_series >= 19) & (hour_series <= 23)) | ((hour_series >= 0) & (hour_series < 2)) # 19:00 to 02:00
    ]
    choices = ['Slot_12_to_18', 'Slot_18_to_19', 'Base_Price_Hours']
    # Use np.select to assign time slots
    return np.select(conditions, choices, default='Outside_Slots') # Default case if none match

In [1057]:
# Assign time slots and sale date
item_df['Time_Slot'] = assign_time_slot_vectorized(item_df['Date'].dt.hour)
item_df[item_df["Time_Slot"] == 'Outside_Slots']

,Date,Price,Time_Slot


In [1058]:

item_df = item_df[item_df["Time_Slot"] != 'Outside_Slots']
item_df['Sale_Date'] = item_df['Date'].dt.date
item_df

,Date,Price,Time_Slot,Sale_Date
0,2021-09-09 21:15:59,24.0,Base_Price_Hours,2021-09-09
1,2021-09-09 22:53:35,24.0,Base_Price_Hours,2021-09-09
2,2021-10-21 21:19:05,24.0,Base_Price_Hours,2021-10-21
3,2021-11-19 14:36:54,14.4,Slot_12_to_18,2021-11-19
4,2022-04-06 15:17:36,14.4,Slot_12_to_18,2022-04-06
5,2022-04-06 16:10:41,14.4,Slot_12_to_18,2022-04-06
6,2022-04-09 16:32:33,14.4,Slot_12_to_18,2022-04-09
7,2022-04-13 16:03:33,14.4,Slot_12_to_18,2022-04-13
8,2022-04-13 17:00:58,14.4,Slot_12_to_18,2022-04-13
9,2022-04-13 17:57:40,14.4,Slot_12_to_18,2022-04-13


In [1059]:
def create_robust_mode_aggregator(agg_func_name: str):
    """
    Factory function that returns a robust mode aggregation function.

    Args:
        agg_func_name (str): The name of the final aggregation to apply on the modes
                             (e.g., 'min' or 'max').

    Returns:
        A function that can be used in an aggfunc.
    """
    def robust_mode_agg(series: pd.Series) -> float :
        """Calculates the mode, then applies a final aggregation (min/max)."""
        modes = series.mode()
        # If multiple modes exist, apply the specified aggregation function
        if len(modes) > 1:
            print(f"Multiple modes found for series with {agg_func_name}:\n{series}\nModes: {modes.tolist()}")
        # If multiple modes exist, return the smallest mode
        if not modes.empty:
            return getattr(modes, agg_func_name)()
        # If no mode found, return NaN
        return np.nan

    robust_mode_agg.__name__ = f'robust_{agg_func_name}_mode'
    return robust_mode_agg


pivot_result = item_df.pivot_table(
    index='Sale_Date',
    columns='Time_Slot',
    values='Price',
    aggfunc=[
        create_robust_mode_aggregator('min'),
        create_robust_mode_aggregator('max')
    ]
)

daily_prices_simple = pd.DataFrame({
    'Slot_12_to_18':  pivot_result[('robust_min_mode', 'Slot_12_to_18')],
    'Slot_18_to_19':  pivot_result[('robust_max_mode', 'Slot_18_to_19')],
    'Base_Price_Hours': pivot_result[('robust_max_mode', 'Base_Price_Hours')]
})


daily_prices = daily_prices_simple.fillna(0).reset_index()  
daily_prices

KeyError: ('robust_max_mode', 'Slot_18_to_19')

In [ ]:
# Create a complete date range from start_date to end_date
all_dates_index = pd.date_range(start=start_date, end=end_date, name='Sale_Date')
all_dates_index

DatetimeIndex(['2021-08-19', '2021-08-20', '2021-08-21', '2021-08-22',
               '2021-08-23', '2021-08-24', '2021-08-25', '2021-08-26',
               '2021-08-27', '2021-08-28',
               ...
               '2025-07-13', '2025-07-14', '2025-07-15', '2025-07-16',
               '2025-07-17', '2025-07-18', '2025-07-19', '2025-07-20',
               '2025-07-21', '2025-07-22'],
              dtype='datetime64[ns]', name='Sale_Date', length=1434, freq='D')

In [ ]:
daily_prices = daily_prices.set_index('Sale_Date').reindex(all_dates_index).fillna(0).reset_index()
daily_prices_org = daily_prices.copy()
daily_prices

,Sale_Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours
0,2021-08-19,0.00,0.00,2.50
1,2021-08-20,0.00,0.00,0.00
2,2021-08-21,0.00,0.00,0.00
3,2021-08-22,0.00,0.00,0.00
4,2021-08-23,0.00,0.00,0.00
...,...,...,...,...
1429,2025-07-18,0.00,0.00,2.75
1430,2025-07-19,2.75,2.75,2.75
1431,2025-07-20,0.00,2.75,2.75
1432,2025-07-21,2.75,2.75,2.75


In [ ]:
def get_mode_excluding_zeros(series: pd.Series) -> float:
    """
    Calculates the mode of a series after filtering out zeros.
    Returns the first mode if it exists, otherwise returns NaN.
    This is a robust and readable alternative to a complex lambda.
    """
    modes = series[series != 0].mode()
    # if length of modes > 1 , print modes
    if len(modes) > 1:
        print("Multiple modes found:", modes.tolist())
    # if length of modes > 1 , return the smallest mode 
    if not modes.empty:
        return modes.min()  # Return the smallest mode
    else:
        return np.nan

In [ ]:
price_columns = daily_prices.columns.drop('Sale_Date')
daily_prices["Month"] = pd.to_datetime(daily_prices['Sale_Date']).dt.to_period('M')

monthly_prices = daily_prices.groupby('Month').agg({col: get_mode_excluding_zeros for col in price_columns}).ffill().bfill().reset_index()
monthly_prices

,Month,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours
0,2021-08,2.50,2.50,2.50
1,2021-09,2.50,2.50,2.50
2,2021-10,2.50,2.50,2.50
3,2021-11,2.50,2.50,2.50
4,2021-12,2.50,2.50,2.50
5,2022-01,2.50,2.50,2.50
6,2022-02,2.50,2.50,2.50
7,2022-03,2.50,2.50,2.50
8,2022-04,2.50,2.50,2.50
9,2022-05,2.50,2.50,2.50


In [ ]:
daily_prices = daily_prices.merge(monthly_prices, on='Month', suffixes=('', '_Monthly_Mode')).drop(columns=['Month']).assign(**{col: lambda x, col=col: np.where(x[col] == 0, x[f'{col}_Monthly_Mode'], x[col]) for col in price_columns})
daily_prices.drop(columns=[f'{col}_Monthly_Mode' for col in price_columns], inplace=True)
daily_prices

,Sale_Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours
0,2021-08-19,2.50,2.50,2.50
1,2021-08-20,2.50,2.50,2.50
2,2021-08-21,2.50,2.50,2.50
3,2021-08-22,2.50,2.50,2.50
4,2021-08-23,2.50,2.50,2.50
...,...,...,...,...
1429,2025-07-18,2.75,2.75,2.75
1430,2025-07-19,2.75,2.75,2.75
1431,2025-07-20,2.75,2.75,2.75
1432,2025-07-21,2.75,2.75,2.75


In [ ]:
def determine_happy_hour_status(df: pd.DataFrame) -> pd.Series:
    """
    Determines the happy hour status for each row in a DataFrame using a fast,
    vectorized approach with np.select.

    Args:
        df (pd.DataFrame): The input DataFrame, must contain the columns
                           'Base_Price_Hours', 'Slot_12_to_18', and 'Slot_18_to_19'.

    Returns:
        pd.Series: A Series containing the calculated happy hour status for each row.
    """
    conditions = [
        (df["Slot_12_to_18"] == df["Base_Price_Hours"]) & (df["Slot_18_to_19"] == df["Base_Price_Hours"]) & (df["Base_Price_Hours"] != 0),
        (df["Slot_18_to_19"] == df["Slot_12_to_18"]) & (df["Slot_18_to_19"] != df["Base_Price_Hours"]),
        (df["Slot_18_to_19"] == df["Base_Price_Hours"]) & (df["Slot_18_to_19"] != df["Slot_12_to_18"]) & (df["Slot_12_to_18"] > df["Base_Price_Hours"] *0.49)
        ]

    choices = [
        "No Happy Hour" ,
        "Happy Hour 12-7",
        "Happy Hour 12-6"
    ]

    return pd.Series(np.select(conditions, choices, default='Unknown'), index=df.index) 

In [ ]:
daily_prices = daily_prices.assign(Happy_Hour_Item=determine_happy_hour_status(daily_prices))
daily_prices["HH_Org"] = daily_prices["Happy_Hour_Item"]
daily_prices

,Sale_Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,Happy_Hour_Item,HH_Org
0,2021-08-19,2.50,2.50,2.50,No Happy Hour,No Happy Hour
1,2021-08-20,2.50,2.50,2.50,No Happy Hour,No Happy Hour
2,2021-08-21,2.50,2.50,2.50,No Happy Hour,No Happy Hour
3,2021-08-22,2.50,2.50,2.50,No Happy Hour,No Happy Hour
4,2021-08-23,2.50,2.50,2.50,No Happy Hour,No Happy Hour
...,...,...,...,...,...,...
1429,2025-07-18,2.75,2.75,2.75,No Happy Hour,No Happy Hour
1430,2025-07-19,2.75,2.75,2.75,No Happy Hour,No Happy Hour
1431,2025-07-20,2.75,2.75,2.75,No Happy Hour,No Happy Hour
1432,2025-07-21,2.75,2.75,2.75,No Happy Hour,No Happy Hour


In [ ]:
# select only the columns Sale_Date, Slot_12_to_18, Slot_18_to_19, Base_Price_Hours , HH_Org, Happy_Hour
daily_prices = daily_prices[['Sale_Date', 'Slot_12_to_18', 'Slot_18_to_19', 'Base_Price_Hours','HH_Org', 'Happy_Hour_Item']]
daily_prices

,Sale_Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,HH_Org,Happy_Hour_Item
0,2021-08-19,2.50,2.50,2.50,No Happy Hour,No Happy Hour
1,2021-08-20,2.50,2.50,2.50,No Happy Hour,No Happy Hour
2,2021-08-21,2.50,2.50,2.50,No Happy Hour,No Happy Hour
3,2021-08-22,2.50,2.50,2.50,No Happy Hour,No Happy Hour
4,2021-08-23,2.50,2.50,2.50,No Happy Hour,No Happy Hour
...,...,...,...,...,...,...
1429,2025-07-18,2.75,2.75,2.75,No Happy Hour,No Happy Hour
1430,2025-07-19,2.75,2.75,2.75,No Happy Hour,No Happy Hour
1431,2025-07-20,2.75,2.75,2.75,No Happy Hour,No Happy Hour
1432,2025-07-21,2.75,2.75,2.75,No Happy Hour,No Happy Hour


In [ ]:
def assign_Discounted_Price(df: pd.DataFrame) -> pd.Series:
    """
    Assigns the discounted price percentage based on happy hour status.
    Args:
        df (pd.DataFrame): The input DataFrame, must contain the columns
                           'Base_Price_Hours', 'Slot_12_to_18', 'Slot_18_to_19', and 'Happy_Hour_Item'.
    Returns:
        pd.Series: A Series containing the calculated discounted price percentage for each row.

    """

    base_price_safe = df['Base_Price_Hours'].replace(0, np.nan)
            
    conditions = [
                df['Happy_Hour_Item'] == 'No Happy Hour',
                df['Happy_Hour_Item'] == 'Happy Hour 12-6',
                df['Happy_Hour_Item'] == 'Happy Hour 12-7'
            ]

    choices = [
                0.0,
                ((base_price_safe - df['Slot_12_to_18']) / base_price_safe) * 100,
                ((base_price_safe - df['Slot_18_to_19']) / base_price_safe) * 100
            ]

    return pd.Series(np.select(conditions, choices, default=0.0), index=df.index).round(2)



In [ ]:
daily_prices = daily_prices.assign(Discount_Item=assign_Discounted_Price(daily_prices))
daily_prices["D_Org"] = daily_prices["Discount_Item"]
daily_prices

,Sale_Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,HH_Org,Happy_Hour_Item,Discount_Item,D_Org
0,2021-08-19,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0
1,2021-08-20,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0
2,2021-08-21,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0
3,2021-08-22,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0
4,2021-08-23,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0
...,...,...,...,...,...,...,...,...
1429,2025-07-18,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0
1430,2025-07-19,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0
1431,2025-07-20,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0
1432,2025-07-21,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0


In [ ]:
daily_prices["Sale_Date"] = pd.to_datetime(daily_prices["Sale_Date"])
daily_prices_org["Sale_Date"] = pd.to_datetime(daily_prices_org["Sale_Date"])
daily_prices_final = pd.merge(daily_prices, daily_prices_org, on='Sale_Date', how='left', suffixes=('', '_Org'))
daily_prices_final

,Sale_Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,HH_Org,Happy_Hour_Item,Discount_Item,D_Org,Slot_12_to_18_Org,Slot_18_to_19_Org,Base_Price_Hours_Org
0,2021-08-19,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0,0.00,0.00,2.50
1,2021-08-20,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0,0.00,0.00,0.00
2,2021-08-21,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0,0.00,0.00,0.00
3,2021-08-22,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0,0.00,0.00,0.00
4,2021-08-23,2.50,2.50,2.50,No Happy Hour,No Happy Hour,0.0,0.0,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...
1429,2025-07-18,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,0.00,0.00,2.75
1430,2025-07-19,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,2.75,2.75,2.75
1431,2025-07-20,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,0.00,2.75,2.75
1432,2025-07-21,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,2.75,2.75,2.75


In [ ]:
daily_prices_final = daily_prices_final[['Sale_Date', 'Slot_12_to_18', 'Slot_18_to_19', "Base_Price_Hours",
                                         "Slot_12_to_18_Org", "Slot_18_to_19_Org", "Base_Price_Hours_Org", "HH_Org",'Happy_Hour_Item','D_Org', 'Discount_Item']]
    

In [ ]:
# rename columns Sale_Date to Date , Happy_Hour_Discount_Percentage to Discount_Item
daily_prices_final = daily_prices_final.rename(columns={"Sale_Date": "Date"})
daily_prices_final["Date"] = pd.to_datetime(daily_prices_final["Date"])

In [ ]:
daily_prices_final = pd.merge(daily_prices_final, df_happy_hour, on='Date', how='left')
# convert Date to date format only (yyyy-mm-dd)
daily_prices_final["Date"] = daily_prices_final["Date"].dt.date
daily_prices_final

,Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,Slot_12_to_18_Org,Slot_18_to_19_Org,Base_Price_Hours_Org,HH_Org,Happy_Hour_Item,D_Org,Discount_Item,Happy_Hour,Discount
0,2021-08-19,2.50,2.50,2.50,0.00,0.00,2.50,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40
1,2021-08-20,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40
2,2021-08-21,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40
3,2021-08-22,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40
4,2021-08-23,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1429,2025-07-18,2.75,2.75,2.75,0.00,0.00,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40
1430,2025-07-19,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40
1431,2025-07-20,2.75,2.75,2.75,0.00,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40
1432,2025-07-21,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40


In [ ]:
ORG_PRICE_COLS = ["Slot_18_to_19_Org", "Slot_12_to_18_Org", "Base_Price_Hours_Org"]
COLS_TO_UPDATE = ["Base_Price_Hours", "Slot_12_to_18", "Slot_18_to_19", "Happy_Hour_Item", "Discount_Item", "HH_Org", "D_Org"]

In [ ]:
# base_price_series = daily_prices_final["Base_Price_Hours_Org"].replace(0, np.nan)
# if not base_price_series.mode().empty:
#     base_price_mode = base_price_series.mode()[0]
# else:
#     base_price_mode = 0

In [ ]:
# def get_mode_excluding_zeros(series: pd.Series) -> float :
#     """
#     Calculates the mode of a series after filtering out zeros.
#     Returns the first mode if it exists, otherwise returns NaN.
#     """
#     filtered_series = series[series != 0]
#     modes = filtered_series.mode()
    
#     if not modes.empty:
#         return modes.iloc[0] 
#     return np.nan



# def add_previous_5_rows_mode(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Adds a new column 'Base_Price_Mode' containing the mode of the 5 PREVIOUS rows
#     of 'Base_Price_Hours', excluding zeros.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: A NEW DataFrame with the added column.
#     """
#     return df.assign(
#         Base_Price_Mode = (
#             df["Base_Price_Hours_Org"]
#             .shift(1)  
#             .rolling(window=5, min_periods=1)  
#             .apply(get_mode_excluding_zeros, raw=False) 
#         )
#     )
# def add_next_5_rows_mode(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Adds a new column 'Base_Price_Mode_Next' containing the mode of the 5 NEXT rows
#     of 'Base_Price_Hours', excluding zeros.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: A NEW DataFrame with the added column.
#     """
#     return df.assign(
#         Base_Price_Mode_Next = (
#             df["Base_Price_Hours_Org"]
#             .shift(-5)                    
#             .rolling(window=5, min_periods=1)
#             .apply(get_mode_excluding_zeros, raw=False)
#         )
#     )

# daily_prices_final = add_previous_5_rows_mode(daily_prices_final)
# daily_prices_final = add_next_5_rows_mode(daily_prices_final)

# daily_prices_final

In [ ]:
def fast_mode_excluding_zeros(arr: np.ndarray) -> float:
    """
    Ultra-fast mode calculation using pure numpy.
    Optimized for small rolling windows.
    """
    # Remove zeros and filter valid values
    non_zero = arr[(arr != 0) & (~np.isnan(arr))]
    
    if len(non_zero) == 0:
        return np.nan
    
    # Use bincount for integer arrays, otherwise use unique
    if np.issubdtype(non_zero.dtype, np.integer):
        counts = np.bincount(non_zero.astype(int))
        return np.argmax(counts)
    else:
        unique_values, counts = np.unique(non_zero, return_counts=True)
        return unique_values[np.argmax(counts)]


def add_mode_columns_optimized(df: pd.DataFrame, 
                              base_column: str = "Base_Price_Hours_Org",
                              window_size: int = 5) -> pd.DataFrame:
    """
    Highly optimized version for large datasets.
    """
    result_df = df.copy()
    values = result_df[base_column].values
    
    # Pre-allocate arrays for results
    n = len(values)
    prev_mode = np.full(n, np.nan)
    next_mode = np.full(n, np.nan)
    
    # Calculate previous mode
    for i in range(n):
        start = max(0, i - window_size )
        window_values = values[start:i]
        prev_mode[i] = fast_mode_excluding_zeros(window_values)
    
    # Calculate next mode
    for i in range(n):
        end = min(n, i + window_size + 1)
        window_values = values[i+1:end]
        next_mode[i] = fast_mode_excluding_zeros(window_values)
    
    result_df["Base_Price_Mode_Opt"] = prev_mode
    result_df["Base_Price_Mode_Next_Opt"] = next_mode
    
    return result_df

In [ ]:
def vectorized_mode_calculation(df: pd.DataFrame, 
                               base_column: str = "Base_Price_Hours_Org",
                               window_size: int = 5) -> pd.DataFrame:
    """
    Vectorized approach for better performance with medium-sized datasets.
    """
    result_df = df.copy()
    values = result_df[base_column].values
    
    # Create rolling windows using stride_tricks (for performance)
    def create_rolling_windows(arr, window):
        shape = (arr.size - window + 1, window)
        strides = (arr.strides[0], arr.strides[0])
        return np.lib.stride_tricks.as_strided(arr, shape=shape, strides=strides)
    
    # Calculate modes using pandas rolling (simpler but may be faster for some cases)
    result_df["Base_Price_Mode"] = (
        result_df[base_column]
        .shift(1)
        .rolling(window=window_size, min_periods=1)
        .apply(lambda x: fast_mode_excluding_zeros(x.values), raw=False)
    )
    
    result_df["Base_Price_Mode_Next"] = (
        result_df[base_column]
        .shift(-window_size)
        .rolling(window=window_size, min_periods=1)
        .apply(lambda x: fast_mode_excluding_zeros(x.values), raw=False)
    )
    
    return result_df

In [ ]:
def hybrid_mode_calculation(df: pd.DataFrame, 
                           base_column: str = "Base_Price_Hours_Org",
                           window_size: int = 5) -> pd.DataFrame:
    """
    Hybrid approach - uses pandas rolling but optimizes the mode function.
    This is typically the FASTEST approach.
    
    Args:
        df: Input DataFrame containing the data
        base_column: Name of the column to calculate mode from (default: "Base_Price_Hours_Org")
        window_size: Size of the rolling window (default: 5)
        
    Returns:
        DataFrame with two new columns: Base_Price_Mode_Hybrid and Base_Price_Mode_Next_Hybrid
    """
    # 1. إنشاء نسخة من DataFrame لحماية البيانات الأصلية
    result_df = df.copy()
    
    # 2. تعريف دالة محسنة لحساب المنوال (القيمة الأكثر تكراراً)
    def optimized_rolling_mode(x):
        """
        الدالة التي ستنفذ على كل نافذة متدحرجة
        x: مصفوفة numpy تحتوي على قيم النافذة الحالية
        """
        # تحويل البيانات إلى مصفوفة numpy (سريعة)
        values = x  # x هو already numpy array بسبب raw=True
        
        # 3. تصفية القيم: إزالة الأصفار والقيم غير المعرفة (NaN)
        non_zero_mask = (values != 0) & (~np.isnan(values))
        non_zero_values = values[non_zero_mask]
        
        # 4. إذا لم يتبقى أي قيم بعد التصفية، نرجع NaN
        if len(non_zero_values) == 0:
            return np.nan
        
        # 5. حساب القيم الفريدة وتكراراتها
        unique_values, counts = np.unique(non_zero_values, return_counts=True)
        
        # 6. إذا وجدنا قيم فريدة، نرجع القيمة الأكثر تكراراً
        if len(unique_values) > 0:
            return unique_values[np.argmax(counts)]  # argmax يعيد موقع أعلى تكرار
        else:
            return np.nan
    
    # 7. حساب المنوال للنافذة السابقة (Previous Window Mode)
    result_df["Base_Price_Mode"] = (
        result_df[base_column]
        .rolling(window=window_size, min_periods=1)  # ← نافذة متدحرجة
        .apply(optimized_rolling_mode, raw=True)  # ← المفتاح السحري للأداء
    )
    
    # 8. حساب المنوال للنافذة التالية (Next Window Mode)
    result_df["Base_Price_Mode_Next"] = (
        result_df[base_column]
        .shift(-window_size+1)  # ← الإزاحة للأمام بمقدار window_size (ننظر للبيانات التالية)
        .rolling(window=window_size, min_periods=1)
        .apply(optimized_rolling_mode, raw=True)
    )
    
    return result_df

In [ ]:
def hybrid_mode_calculation_ultra(df: pd.DataFrame, 
                                base_column: str = "Base_Price_Hours_Org",
                                window_size: int = 5) -> pd.DataFrame:
    """
    Ultra-optimized version for very large datasets.
    """
    result_df = df.copy()
    
    # استخدام numpy مباشرة للحصول على أفضل أداء
    values = result_df[base_column].values
    
    # حساب القيم المزاحة باستخدام numpy (أسرع من pandas shift)
    shifted_prev = np.empty_like(values)
    shifted_prev[0] = np.nan
    shifted_prev[1:] = values[:-1]
    
    shifted_next = np.empty_like(values)
    shifted_next[-window_size:] = np.nan
    shifted_next[:-window_size] = values[window_size:]
    
    # تخزين كأعمدة مؤقتة
    result_df["_shifted_prev"] = shifted_prev
    result_df["_shifted_next"] = shifted_next
    
    # باقي الكود يبقى كما هو...
    def optimized_rolling_mode(x):
        values = x
        non_zero_mask = (values != 0) & (~np.isnan(values))
        non_zero_values = values[non_zero_mask]
        if len(non_zero_values) == 0:
            return np.nan
        unique_values, counts = np.unique(non_zero_values, return_counts=True)
        return unique_values[np.argmax(counts)] if len(unique_values) > 0 else np.nan
    
    result_df["Base_Price_Mode"] = (
        result_df["_shifted_prev"]
        .rolling(window=window_size, min_periods=1)
        .apply(optimized_rolling_mode, raw=True)
    )
    
    result_df["Base_Price_Mode_Next"] = (
        result_df["_shifted_next"]
        .rolling(window=window_size, min_periods=1)
        .apply(optimized_rolling_mode, raw=True)
    )
    
    result_df.drop(["_shifted_prev", "_shifted_next"], axis=1, inplace=True)
    
    return result_df

# result = hybrid_mode_calculation_ultra(daily_prices_final.copy())
# result

In [ ]:
import numpy as np
import pandas as pd
from numba import njit

# --- Numba-accelerated Mode function ---
@njit
def rolling_mode_window(values, window_back, window_forward):
    """
    Ultra-fast rolling mode using numba.
    Calculates the most frequent non-zero & non-NaN value
    in a sliding window of (back + current + forward).
    """
    n = len(values)
    out = np.empty(n, dtype=np.float64)
    
    # temporary buffer (local array) to collect window values
    buffer = np.empty(window_back + window_forward + 1, dtype=np.float64)
    
    for i in range(n):
        start = max(0, i - window_back)
        end = min(n, i + window_forward + 1)
        size = 0  # actual number of valid entries
        
        # collect values into local buffer
        for j in range(start, end):
            v = values[j]
            if not np.isnan(v) and v != 0:
                buffer[size] = v
                size += 1
        
        if size == 0:
            out[i] = np.nan
            continue
        
        # compute mode manually (numba no support for numpy.unique with counts)
        # simple O(n^2) but very fast since window size is tiny (=7)
        best_value = buffer[0]
        best_count = 1
        
        for a in range(size):
            count = 1
            for b in range(a + 1, size):
                if buffer[a] == buffer[b]:
                    count += 1
            if count > best_count:
                best_count = count
                best_value = buffer[a]
        
        out[i] = best_value
    
    return out


# --- Public function you will use ---
def add_base_price_mode_numba(
        df: pd.DataFrame,
        base_column: str = "Base_Price_Hours_Org",
        window_back: int = 3,
        window_forward: int = 3) -> pd.DataFrame:
    """
    Ultra-fast rolling mode using Numba (20x-40x faster).
    Computes mode for a window of (back + current + forward).
    """
    
    df_out = df.copy()
    values = df_out[base_column].astype(float).to_numpy()

    result = rolling_mode_window(values, window_back, window_forward)
    df_out["Base_Price_Mode_V"] = result
    
    return df_out




In [ ]:
def calculate_base_price_mode_vectorized(df: pd.DataFrame, 
                                       base_column: str = "Base_Price_Hours_Org",
                                       window_before: int = 3,
                                       window_after: int = 3) -> pd.DataFrame:
    """
    Vectorized implementation for maximum performance.
    """
    result_df = df.copy()
    values = result_df[base_column].values
    n = len(values)
    
    # Create sliding windows using numpy strides
    def sliding_window(arr, window_size, step_size=1):
        """Create sliding windows of specified size"""
        num_windows = (len(arr) - window_size) // step_size + 1
        shape = (num_windows, window_size)
        strides = (arr.strides[0] * step_size, arr.strides[0])
        return np.lib.stride_tricks.as_strided(arr, shape=shape, strides=strides)
    
    total_window_size = window_before + window_after + 1
    windows = sliding_window(values, total_window_size)
    
    # Calculate mode for each window
    def vectorized_mode(windows_arr):
        """Vectorized mode calculation for multiple windows"""
        results = np.full(windows_arr.shape[0], np.nan)
        
        for i, window in enumerate(windows_arr):
            non_zero_mask = (window != 0) & (~np.isnan(window))
            non_zero_values = window[non_zero_mask]
            
            if len(non_zero_values) > 0:
                unique_values, counts = np.unique(non_zero_values, return_counts=True)
                results[i] = unique_values[np.argmax(counts)]
        
        return results
    
    # Calculate modes and handle edge cases
    window_modes = vectorized_mode(windows)
    
    # Pad the results to match original length
    pad_left = window_before
    pad_right = n - len(window_modes) - pad_left
    
    base_price_mode = np.pad(
        window_modes, 
        (pad_left, pad_right), 
        mode='constant', 
        constant_values=np.nan
    )
    
    result_df["Base_Price_Mode_V"] = base_price_mode
    return result_df

In [ ]:
def fastest_mode_calculation(df: pd.DataFrame, 
                           base_column: str = "Base_Price_Hours_Org",
                           window_size: int = 5) -> pd.DataFrame:
    """
    Fastest implementation - 2.75x faster than vectorized approach!
    Based on actual benchmark results.
    """
    result_df = df.copy()
    
    # 1. Precompute shifted columns - BIGGEST performance gain
    prev_shifted = result_df[base_column].shift(1)
    next_shifted = result_df[base_column].shift(-window_size)
    
    # 2. Ultra-optimized mode function
    def ultra_fast_mode(x):
        """
        Optimized mode calculation using pure numpy.
        Key optimizations:
        - Direct numpy array operations
        - Efficient zero and NaN filtering
        - bincount for integers (much faster)
        - Minimal memory allocations
        """
        # Convert to numpy array (already done with raw=True, but safe check)
        arr = x if isinstance(x, np.ndarray) else np.asarray(x)
        
        # Fast filtering using boolean indexing
        mask = (arr != 0) & (~np.isnan(arr))
        non_zero_values = arr[mask]
        
        if len(non_zero_values) == 0:
            return np.nan
        
        # Use bincount for integer arrays (3-5x faster than unique)
        if np.issubdtype(non_zero_values.dtype, np.integer):
            try:
                counts = np.bincount(non_zero_values.astype(int))
                if len(counts) > 0:
                    return np.argmax(counts).astype(float)
            except (ValueError, IndexError):
                # Fallback for edge cases
                pass
        
        # Fallback for float arrays or bincount failures
        unique_values, counts = np.unique(non_zero_values, return_counts=True)
        return unique_values[np.argmax(counts)] if len(unique_values) > 0 else np.nan
    
    # 3. Apply with raw=True for maximum performance
    result_df["Base_Price_Mode"] = (
        prev_shifted
        .rolling(window=window_size, min_periods=1)
        .apply(ultra_fast_mode, raw=True)  # raw=True passes numpy arrays directly
    )
    
    result_df["Base_Price_Mode_Next"] = (
        next_shifted
        .rolling(window=window_size, min_periods=1) 
        .apply(ultra_fast_mode, raw=True)
    )
    
    return result_df

In [ ]:
import time


def benchmark_methods():

    start_time = time.time()
    result1=vectorized_mode_calculation(daily_prices_final.copy())
    time1 = time.time() - start_time


    start_time = time.time()
    result2 = hybrid_mode_calculation(daily_prices_final.copy())  
    time2 = time.time() - start_time
    
   
    start_time = time.time()
    result3 = hybrid_mode_calculation_ultra(daily_prices_final.copy())
    time3 = time.time() - start_time

    start_time = time.time()
    result4 = fastest_mode_calculation(daily_prices_final.copy())
    time4 = time.time() - start_time


    print(f"vectorized_mode_calculation: {time1:.4f} seconds")
    print(f"hybrid_mode_calculation: {time2:.4f} seconds")
    print(f"hybrid_mode_calculation_ultra: {time3:.4f} seconds")
    print(f"fastest_mode_calculation: {time4:.4f} seconds")
   

    

benchmark_methods()

vectorized_mode_calculation: 0.0638 seconds
hybrid_mode_calculation: 0.0317 seconds
hybrid_mode_calculation_ultra: 0.0358 seconds
fastest_mode_calculation: 0.0346 seconds


In [ ]:
import time


def benchmark_methods():

    start_time = time.time()
    result1=add_base_price_mode_numba(daily_prices_final.copy())
    time1=time.time() - start_time

    start_time = time.time()
    result2=calculate_base_price_mode_vectorized(daily_prices_final.copy())
    time2=time.time() - start_time


    print(f"add_base_price_mode_numba: {time1:.4f} seconds")
    print(f"calculate_base_price_mode_vectorized: {time2:.4f} seconds")
   

    

benchmark_methods()

add_base_price_mode_numba: 0.6781 seconds
calculate_base_price_mode_vectorized: 0.0157 seconds


In [ ]:
daily_prices_final = hybrid_mode_calculation(daily_prices_final.copy())
daily_prices_final

,Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,Slot_12_to_18_Org,Slot_18_to_19_Org,Base_Price_Hours_Org,HH_Org,Happy_Hour_Item,D_Org,Discount_Item,Happy_Hour,Discount,Base_Price_Mode,Base_Price_Mode_Next
0,2021-08-19,2.50,2.50,2.50,0.00,0.00,2.50,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN
1,2021-08-20,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN
2,2021-08-21,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN
3,2021-08-22,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,2.50
4,2021-08-23,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,2.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1429,2025-07-18,2.75,2.75,2.75,0.00,0.00,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75
1430,2025-07-19,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75
1431,2025-07-20,2.75,2.75,2.75,0.00,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75
1432,2025-07-21,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75


In [ ]:
daily_prices_final = add_base_price_mode_numba(daily_prices_final.copy())
daily_prices_final

,Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,Slot_12_to_18_Org,Slot_18_to_19_Org,Base_Price_Hours_Org,HH_Org,Happy_Hour_Item,D_Org,Discount_Item,Happy_Hour,Discount,Base_Price_Mode,Base_Price_Mode_Next,Base_Price_Mode_V
0,2021-08-19,2.50,2.50,2.50,0.00,0.00,2.50,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50
1,2021-08-20,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50
2,2021-08-21,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50
3,2021-08-22,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,2.50,2.50
4,2021-08-23,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,2.50,2.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1429,2025-07-18,2.75,2.75,2.75,0.00,0.00,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75
1430,2025-07-19,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75
1431,2025-07-20,2.75,2.75,2.75,0.00,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75
1432,2025-07-21,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75


In [ ]:
# df_processed = daily_prices_final.assign(
#     is_Cost_Price = lambda df: (
#         ((df["Slot_18_to_19_Org"] < (df['Base_Price_Mode'] * 0.49)) & (df["Slot_18_to_19_Org"] != 0)) |
#         ((df["Slot_12_to_18_Org"] < (df['Base_Price_Mode'] * 0.49)) & (df["Slot_12_to_18_Org"] != 0)) |
#         ((df["Base_Price_Hours_Org"] < (df['Base_Price_Mode'] * 0.49)) & (df["Base_Price_Hours_Org"] != 0))
#     ),
    
#     Cost_Price_Value = lambda df: np.where(
#         df['is_Cost_Price'], 
#         df[ORG_PRICE_COLS].replace(0, np.nan).min(axis=1),
#         np.nan  
#     )
# )

# df_processed

In [ ]:
# # Set specified columns to NaN where is_Cost_Price is True
# cost_price_mask = df_processed['is_Cost_Price']
# df_processed.loc[cost_price_mask, COLS_TO_UPDATE] = np.nan
# df_processed

In [ ]:
# df_processed[COLS_TO_UPDATE + ['Cost_Price_Value']] = df_processed[COLS_TO_UPDATE + ['Cost_Price_Value']].ffill().bfill()

In [ ]:
# if df_processed["Cost_Price_Value"].isnull().all():
#     cost_price_fill_value = (base_price_mode * np.random.uniform(0.3, 0.35)).round(2)
#     df_processed["Cost_Price_Value"] = df_processed["Cost_Price_Value"].fillna(cost_price_fill_value)

In [ ]:
# daily_prices_final = df_processed.drop(columns=["is_Cost_Price"])
# daily_prices_final

In [ ]:
# تعريف الثوابت
ORG_PRICE_COLS = ["Slot_18_to_19_Org", "Slot_12_to_18_Org", "Base_Price_Hours_Org"]
COLS_TO_UPDATE = ["Base_Price_Hours", "Slot_12_to_18", "Slot_18_to_19", "Happy_Hour_Item", "Discount_Item", "HH_Org", "D_Org"]

def process_cost_prices_complete(df: pd.DataFrame) -> pd.DataFrame:
    """
    Complete cost price processing with optimized performance and error handling.
    """
    df_processed = df.copy()
    
    # 1. حساب عتبة Cost Price مرة واحدة فقط
    threshold = df_processed['Base_Price_Mode'] * 0.49
    
    # 2. إنشاء شروط Cost Price بكفاءة
    cost_conditions = []
    for col in ORG_PRICE_COLS:
        condition = (df_processed[col] < threshold) & (df_processed[col] != 0)
        cost_conditions.append(condition)
    
    # 3. دمج الشروط باستخدام OR منطقي
    is_cost_price = cost_conditions[0]
    for condition in cost_conditions[1:]:
        is_cost_price = is_cost_price | condition
    
    df_processed['is_Cost_Price'] = is_cost_price

    if is_cost_price.any():
        print("Cost Price rows detected.")
        # indexes of cost price rows
        cost_price_indexes = df_processed.index[is_cost_price].tolist()
        print("Indexes of Cost Price rows:", cost_price_indexes)
    else:
        print("No Cost Price rows detected.")
    
    # 4. حساب Cost_Price_Value بكفاءة
    price_matrix = df_processed[ORG_PRICE_COLS].copy()
    price_matrix = price_matrix.replace(0, np.nan)
    
    df_processed['Cost_Price_Value'] = np.where(
        is_cost_price,
        price_matrix.min(axis=1),
        np.nan
    )
    
    # 5. تعيين NaN للأعمدة المطلوبة في صفوف Cost Price
    cost_price_mask = df_processed['is_Cost_Price']
    df_processed.loc[cost_price_mask, COLS_TO_UPDATE] = np.nan
    
    # 6. تعبئة القيم المفقودة (Forward then Backward fill)
    fill_columns = COLS_TO_UPDATE + ['Cost_Price_Value']
    df_processed[fill_columns] = df_processed[fill_columns].ffill().bfill()
    
    # 7. معالجة الحالة التي تكون فيها جميع قيم Cost_Price_Value مفقودة
    if df_processed["Cost_Price_Value"].isnull().all():
        # استخدام Base_Price_Mode من DataFrame بدلاً من متغير غير معرف
        base_price_mode = df_processed['Base_Price_Mode']
        cost_price_fill_value = (base_price_mode * np.random.uniform(0.3, 0.35)).round(2)
        df_processed["Cost_Price_Value"] = cost_price_fill_value
    
    # 8. إزالة العمود المؤقت
    df_processed = df_processed.drop(columns=["is_Cost_Price"])
    
    return df_processed

# الاستخدام
daily_prices_final = process_cost_prices_complete(daily_prices_final)
daily_prices_final

Cost Price rows detected.
Indexes of Cost Price rows: [1381]


,Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,Slot_12_to_18_Org,Slot_18_to_19_Org,Base_Price_Hours_Org,HH_Org,Happy_Hour_Item,D_Org,Discount_Item,Happy_Hour,Discount,Base_Price_Mode,Base_Price_Mode_Next,Base_Price_Mode_V,Cost_Price_Value
0,2021-08-19,2.50,2.50,2.50,0.00,0.00,2.50,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50,0.61
1,2021-08-20,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50,0.61
2,2021-08-21,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50,0.61
3,2021-08-22,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,2.50,2.50,0.61
4,2021-08-23,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,2.50,2.50,0.61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1429,2025-07-18,2.75,2.75,2.75,0.00,0.00,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61
1430,2025-07-19,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61
1431,2025-07-20,2.75,2.75,2.75,0.00,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61
1432,2025-07-21,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61


In [ ]:
# # add columns [margin_on_base_price ,profit_on_base_price, margin_on_happy_hour_price, profit_on_happy_hour_price]
# daily_prices_final = daily_prices_final.assign(
#     margin_on_base_price = lambda df: (
#         ((df["Base_Price_Hours"] - df["Cost_Price_Value"]) / df["Base_Price_Hours"]) * 100
#     ).round(2),
#     profit_on_base_price = lambda df: (
#         ((df["Base_Price_Hours"] - df["Cost_Price_Value"])
#     ).round(2)),
#     margin_on_happy_hour_price = lambda df: (
#         ((df["Slot_12_to_18"] - df["Cost_Price_Value"]) / df["Slot_12_to_18"]) * 100
#     ).round(2),
#     profit_on_happy_hour_price = lambda df: (
#         ((df["Slot_12_to_18"] - df["Cost_Price_Value"])
#     ).round(2))
# )
# daily_prices_final

In [ ]:
# --- Constants ---
UNKNOWN_LABEL = "Unknown"
HH_12_7_LABEL = "Happy Hour 12-7"
HH_12_6_LABEL = "Happy Hour 12-6"
NO_HH_LABEL = "No Happy Hour"
VALID_HH_LABELS = [HH_12_7_LABEL, HH_12_6_LABEL]

def apply_correction_engine(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies prioritized correction rules with maximum performance.
    Uses precomputed and pre-merged conditions for optimal efficiency.
    """
    
    df_updated = df.copy()
    df_updated["Case_Applied"] = np.nan  # Fixed: use df_updated instead of df
    
    # --- Precompute all conditions once ---
    hh_org = df_updated["HH_Org"]
    happy_hour = df_updated["Happy_Hour"]
    base_price_hours_org = df_updated["Base_Price_Hours_Org"]
    slot_12_to_18_org = df_updated["Slot_12_to_18_Org"]
    slot_18_to_19_org = df_updated["Slot_18_to_19_Org"]
    base_price_mode = df_updated["Base_Price_Mode"]
    base_price_mode_next = df_updated["Base_Price_Mode_Next"]
    D_Org = df_updated["D_Org"]
    Discount = df_updated["Discount"]
    
    # Precompute all possible conditions
    unknown_hh_Org_with_12_7_hh = (hh_org == UNKNOWN_LABEL) & (happy_hour == HH_12_7_LABEL)
    unknown_hh_Org_with_12_6_hh = (hh_org == UNKNOWN_LABEL) & (happy_hour == HH_12_6_LABEL)
    unknown_hh_Org_with_no_hh = (hh_org == UNKNOWN_LABEL) & (happy_hour == NO_HH_LABEL)
    hh_Org_12_7_with_hh_12_6 = (hh_org == HH_12_7_LABEL) & (happy_hour == HH_12_6_LABEL)
    hh_Org_12_6_with_hh_12_7 = (hh_org == HH_12_6_LABEL) & (happy_hour == HH_12_7_LABEL)
    slot_12_18_Org_zero = (slot_12_to_18_org == 0)
    slot_12_18_Org_non_zero = (slot_12_to_18_org != 0)
    slot_18_19_Org_zero = (slot_18_to_19_org == 0)
    slot_18_19_Org_non_zero = (slot_18_to_19_org != 0)
    base_price_hours_Org_zero = (base_price_hours_org == 0)
    base_price_hours_Org_non_zero = (base_price_hours_org != 0)
    base_price_hours_org_eq_base_price_mode_next = (base_price_hours_org == base_price_mode_next)
    base_price_hours_org_eq_base_price_mode = (base_price_hours_org == base_price_mode)
    base_price_hours_org_eq_slot_18_19_org = (base_price_hours_org == slot_18_to_19_org)
    base_price_hours_org_ne_base_price_mode = (base_price_hours_org != base_price_mode)

    # base price mode 
    base_price_mode_eq_base_price_next = (base_price_mode == base_price_mode_next)

    # discounts conditions
    D_Org_ne_Discount = (D_Org != Discount)

    # Precompute discounted base price mode for Case 19
    discounted_base_price_mode = (base_price_mode * (1 - (Discount / 100))).round(1)
    base_price_hours_org_eq_discounted_base_price_mode = (base_price_hours_org == discounted_base_price_mode)

    def calculate_base_price_from_slot_12_18(df, mask):
        return (df.loc[mask, "Slot_12_to_18_Org"] / (1 - (df.loc[mask, "Discount"] / 100))).round(1)

    def calculate_hh_from_base_price(df, mask):
        return (df.loc[mask, "Base_Price_Hours_Org"] * (1 - (df.loc[mask, "Discount"] / 100))).round(1)
    
    def calculate_hh_from_base_price_mode(df, mask):
        return (df.loc[mask, "Base_Price_Mode"] * (1 - (df.loc[mask, "Discount"] / 100))).round(1)

    # --- Define cases with pre-merged mask expressions ---
    correction_cases = [
        {
            "case_name": "Case 1",
            "mask": (
                unknown_hh_Org_with_12_7_hh &
                base_price_hours_Org_zero & 
                slot_18_19_Org_non_zero &
                slot_12_18_Org_zero & 
                (base_price_mode == base_price_mode_next)
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Slot_12_to_18": calculate_hh_from_base_price_mode,
                "Slot_18_to_19": calculate_hh_from_base_price_mode
            }
        },
        {
            "case_name": "Case 2",
            "mask": (
                unknown_hh_Org_with_12_7_hh &
                base_price_hours_Org_non_zero &
                slot_12_18_Org_non_zero &
                slot_18_19_Org_zero
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Slot_18_to_19": "Slot_12_to_18_Org"
            }
        },
        {
            "case_name": "Case 3", 
            "mask": (
                unknown_hh_Org_with_12_7_hh &
                base_price_hours_Org_zero &
                slot_12_18_Org_non_zero &
                slot_18_19_Org_zero
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour", 
                "Discount_Item": "Discount",
                "Slot_18_to_19": "Slot_12_to_18_Org"
            }
        },
        {
            "case_name": "Case 4",
            "mask": (
                unknown_hh_Org_with_12_7_hh &
                base_price_hours_Org_non_zero &
                slot_12_18_Org_zero &
                slot_18_19_Org_non_zero &
                (base_price_hours_org != slot_18_to_19_org)
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount", 
                "Slot_12_to_18": calculate_hh_from_base_price,
                "Slot_18_to_19": calculate_hh_from_base_price
            }
        },
        {
            "case_name": "Case 5",
            "mask": hh_Org_12_7_with_hh_12_6,
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Slot_18_to_19": "Base_Price_Hours"
            }
        },
        {
            "case_name": "Case 6",
            "mask": (unknown_hh_Org_with_no_hh &
            base_price_hours_Org_non_zero &
            slot_12_18_Org_non_zero &
            slot_18_19_Org_zero
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Slot_18_to_19": "Base_Price_Hours"
            }
        },
        {
            "case_name": "Case 7",
            "mask": (unknown_hh_Org_with_12_6_hh &
            base_price_hours_Org_non_zero &
            slot_12_18_Org_zero &
            slot_18_19_Org_zero &
            base_price_hours_org_eq_base_price_mode_next
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Slot_18_to_19": "Base_Price_Hours",
                "Slot_12_to_18": calculate_hh_from_base_price
            }
        },
        {
            "case_name": "Case 8",
            "mask": (unknown_hh_Org_with_12_6_hh &
            base_price_hours_Org_non_zero &
            slot_12_18_Org_non_zero &
            slot_18_19_Org_zero &
            base_price_hours_org_eq_base_price_mode_next
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Slot_18_to_19": "Base_Price_Hours",
            }
        },
        # {
        #     "case_name": "Case 9",
        #     "mask": (D_Org_ne_Discount &
        #     base_price_hours_org_eq_slot_18_19_org &
        #     base_price_hours_Org_non_zero &
        #     slot_12_18_Org_zero 
        #     ),
        #     "updates": {
        #         "Discount_Item": "Discount",
        #         "Slot_12_to_18": calculate_hh_from_base_price
        #     }
        # },
        {
            "case_name": "Case 10",
            "mask": (D_Org_ne_Discount &
            base_price_hours_org_eq_slot_18_19_org &
            base_price_hours_Org_zero &
            slot_12_18_Org_non_zero &
            (hh_org == HH_12_6_LABEL)
            ),
            "updates": {
                "Discount_Item": "Discount",
                "Base_Price_Hours": calculate_base_price_from_slot_12_18,
                "Slot_18_to_19": calculate_base_price_from_slot_12_18
            }
        },
        {
            "case_name": "Case 11",
            "mask": (D_Org_ne_Discount &
            base_price_hours_Org_zero &
            slot_12_18_Org_non_zero &
            base_price_hours_org_eq_slot_18_19_org &
            (hh_org == HH_12_7_LABEL)
            ),
            "updates": {
                "Discount_Item": "Discount",
                "Base_Price_Hours": calculate_base_price_from_slot_12_18,
                "Slot_18_to_19": "Slot_12_to_18_Org"
            }
        },
        {
            "case_name": "Case 12",
            "mask": (D_Org_ne_Discount &
            base_price_hours_Org_non_zero &
            slot_12_18_Org_zero &
            slot_18_19_Org_zero &
            (happy_hour == HH_12_7_LABEL)
            ),
            "updates": {
                "Discount_Item": "Discount",
                "Happy_Hour_Item": "Happy_Hour",
                "Slot_12_to_18": calculate_hh_from_base_price,
                "Slot_18_to_19": calculate_hh_from_base_price
            }
        },
        # {
        #     "case_name": "Case 13",
        #     "mask": (D_Org_ne_Discount &
        #     base_price_hours_Org_non_zero &
        #     slot_12_18_Org_zero &
        #     slot_18_19_Org_zero &
        #     (happy_hour == HH_12_6_LABEL) &
        #     (base_price_mode == base_price_mode_next) &
        #     base_price_hours_org_eq_base_price_mode
        #     ),
        #     "updates": {
        #         "Discount_Item": "Discount",
        #         "Happy_Hour_Item": "Happy_Hour",
        #         "Slot_12_to_18": calculate_hh_from_base_price,
        #         "Slot_18_to_19":"Base_Price_Hours_Org"
        #     }
        # },
        {
            "case_name": "Case 14",
            "mask": (D_Org_ne_Discount &
            base_price_hours_Org_non_zero &
            slot_12_18_Org_zero &
            slot_18_19_Org_zero &
            (happy_hour == HH_12_6_LABEL) &
            base_price_mode_eq_base_price_next &
            base_price_hours_org_ne_base_price_mode
            ),
            "updates": {
                "Discount_Item": "Discount",
                "Happy_Hour_Item": "Happy_Hour",
                "Slot_12_to_18": calculate_hh_from_base_price_mode,
                "Slot_18_to_19": calculate_hh_from_base_price_mode,
                "Base_Price_Hours": "Base_Price_Mode"
            }
        },
        {
            "case_name": "Case 15",
            "mask": ((hh_org.isin([HH_12_6_LABEL, HH_12_7_LABEL])) &
            (happy_hour == NO_HH_LABEL) & 
            base_price_hours_Org_non_zero &
            slot_18_19_Org_zero &
            slot_12_18_Org_zero 
            ),
            "updates": {
                "Discount_Item": "Discount",
                "Happy_Hour_Item": "Happy_Hour",
                "Slot_18_to_19": "Base_Price_Hours",
                "Slot_12_to_18": "Base_Price_Hours"
            }
        },
        {
            "case_name": "Case 16",
            "mask": ((hh_org.isin([HH_12_6_LABEL, HH_12_7_LABEL])) &
            (happy_hour == NO_HH_LABEL) & 
            base_price_hours_Org_non_zero &
            slot_18_19_Org_non_zero &
            slot_12_18_Org_zero 
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Slot_12_to_18": "Base_Price_Hours"
            }
        },
        {
            "case_name": "Case 17",
            "mask": (base_price_hours_Org_zero &
                     slot_18_19_Org_zero &
                     slot_12_18_Org_zero &
                     (base_price_mode == base_price_mode_next) &
                     (happy_hour == HH_12_6_LABEL)
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Base_Price_Hours": "Base_Price_Mode",
                "Slot_18_to_19": "Base_Price_Mode",
                "Slot_12_to_18": calculate_hh_from_base_price_mode
            }
        },
        {
            "case_name": "Case 18",
            "mask": (base_price_hours_Org_zero &
                     slot_18_19_Org_zero &
                     slot_12_18_Org_zero &
                     (base_price_mode == base_price_mode_next) &
                     (happy_hour == HH_12_7_LABEL)
            ),
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Base_Price_Hours": "Base_Price_Mode",
                "Slot_18_to_19": calculate_hh_from_base_price_mode,
                "Slot_12_to_18": calculate_hh_from_base_price_mode
            }
        },
        {
            "case_name": "Case 19",
            "mask": (hh_org == NO_HH_LABEL) &
                    (happy_hour == HH_12_7_LABEL) &
                    (base_price_mode == base_price_mode_next) &
                    base_price_hours_org_eq_discounted_base_price_mode,  # Fixed: use precomputed condition
            "updates": {
                "Happy_Hour_Item": "Happy_Hour",
                "Discount_Item": "Discount",
                "Base_Price_Hours": "Base_Price_Mode",
                "Slot_18_to_19": calculate_hh_from_base_price_mode,
                "Slot_12_to_18": calculate_hh_from_base_price_mode
            }
        }
    ]
    
    # --- Apply corrections with maximum efficiency ---
    for case in correction_cases:
        mask = case["mask"]
        
        if mask.any():
            print(f"Applying {case['case_name']} → {mask.sum()} rows.")
            
            for target_col, source_col in case["updates"].items():
                if callable(source_col):
                    df_updated.loc[mask, target_col] = source_col(df_updated, mask)
                else:
                    df_updated.loc[mask, target_col] = df_updated.loc[mask, source_col]
            
            # Update Case_Applied column
            current_cases = df_updated.loc[mask, "Case_Applied"]
            df_updated.loc[mask, "Case_Applied"] = current_cases.fillna('') + case['case_name'] + ', '
    
    return df_updated

daily_prices_final = apply_correction_engine(daily_prices_final)
daily_prices_final

Applying Case 5 → 1 rows.
Applying Case 12 → 76 rows.
Applying Case 17 → 10 rows.
Applying Case 18 → 5 rows.


C:\Users\fadia\AppData\Local\Temp\ipykernel_32368\1149944327.py:359: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Case 5, ']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_updated.loc[mask, "Case_Applied"] = current_cases.fillna('') + case['case_name'] + ', '


,Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,Slot_12_to_18_Org,Slot_18_to_19_Org,Base_Price_Hours_Org,HH_Org,Happy_Hour_Item,D_Org,Discount_Item,Happy_Hour,Discount,Base_Price_Mode,Base_Price_Mode_Next,Base_Price_Mode_V,Cost_Price_Value,Case_Applied
0,2021-08-19,2.50,2.50,2.50,0.00,0.00,2.50,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50,0.61,NaN
1,2021-08-20,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50,0.61,NaN
2,2021-08-21,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50,0.61,NaN
3,2021-08-22,1.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,Happy Hour 12-6,0.0,40.0,Happy Hour 12-6,40,2.50,2.50,2.50,0.61,"Case 17,"
4,2021-08-23,1.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,Happy Hour 12-6,0.0,40.0,Happy Hour 12-6,40,2.50,2.50,2.50,0.61,"Case 17,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1429,2025-07-18,1.60,1.60,2.75,0.00,0.00,2.75,No Happy Hour,Happy Hour 12-7,0.0,40.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,"Case 12,"
1430,2025-07-19,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,NaN
1431,2025-07-20,2.75,2.75,2.75,0.00,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,NaN
1432,2025-07-21,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,NaN


In [ ]:
# get the rows where Case_Applied is not null
cases_applied_df = daily_prices_final[daily_prices_final["Case_Applied"].notnull()]
cases_applied_df

,Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,Slot_12_to_18_Org,Slot_18_to_19_Org,Base_Price_Hours_Org,HH_Org,Happy_Hour_Item,D_Org,Discount_Item,Happy_Hour,Discount,Base_Price_Mode,Base_Price_Mode_Next,Base_Price_Mode_V,Cost_Price_Value,Case_Applied
3,2021-08-22,1.5,2.5,2.50,0.0,0.0,0.00,No Happy Hour,Happy Hour 12-6,0.0,40.0,Happy Hour 12-6,40,2.50,2.50,2.50,0.61,"Case 17,"
4,2021-08-23,1.5,2.5,2.50,0.0,0.0,0.00,No Happy Hour,Happy Hour 12-6,0.0,40.0,Happy Hour 12-6,40,2.50,2.50,2.50,0.61,"Case 17,"
22,2021-09-10,1.5,2.5,2.50,0.0,0.0,0.00,No Happy Hour,Happy Hour 12-6,0.0,40.0,Happy Hour 12-6,40,2.50,2.50,2.50,0.61,"Case 17,"
38,2021-09-26,1.5,2.5,2.50,0.0,0.0,0.00,No Happy Hour,Happy Hour 12-6,0.0,40.0,Happy Hour 12-6,40,2.50,2.50,2.50,0.61,"Case 17,"
161,2022-01-27,1.5,2.5,2.50,0.0,0.0,0.00,No Happy Hour,Happy Hour 12-6,0.0,40.0,Happy Hour 12-6,40,2.50,2.50,2.50,0.61,"Case 17,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408,2025-06-27,1.6,1.6,2.75,0.0,0.0,2.75,No Happy Hour,Happy Hour 12-7,0.0,40.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,"Case 12,"
1411,2025-06-30,1.6,1.6,2.75,0.0,0.0,2.75,No Happy Hour,Happy Hour 12-7,0.0,40.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,"Case 12,"
1412,2025-07-01,1.6,1.6,2.75,0.0,0.0,2.75,No Happy Hour,Happy Hour 12-7,0.0,40.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,"Case 12,"
1425,2025-07-14,1.6,1.6,2.75,0.0,0.0,2.75,No Happy Hour,Happy Hour 12-7,0.0,40.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,"Case 12,"


In [ ]:
# get rows where happy hour item not equal to happy hour and discount item not equal to discount
discrepancies_df = daily_prices_final[
    (daily_prices_final["Happy_Hour_Item"] != daily_prices_final["Happy_Hour"]) |
    (daily_prices_final["Discount_Item"] != daily_prices_final["Discount"])
]   
discrepancies_df

,Date,Slot_12_to_18,Slot_18_to_19,Base_Price_Hours,Slot_12_to_18_Org,Slot_18_to_19_Org,Base_Price_Hours_Org,HH_Org,Happy_Hour_Item,D_Org,Discount_Item,Happy_Hour,Discount,Base_Price_Mode,Base_Price_Mode_Next,Base_Price_Mode_V,Cost_Price_Value,Case_Applied
0,2021-08-19,2.50,2.50,2.50,0.00,0.00,2.50,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50,0.61,NaN
1,2021-08-20,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50,0.61,NaN
2,2021-08-21,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,2.50,NaN,2.50,0.61,NaN
5,2021-08-24,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,NaN,2.50,2.50,0.61,NaN
6,2021-08-25,2.50,2.50,2.50,0.00,0.00,0.00,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-6,40,NaN,2.50,2.50,0.61,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428,2025-07-17,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,NaN
1430,2025-07-19,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,NaN
1431,2025-07-20,2.75,2.75,2.75,0.00,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,NaN
1432,2025-07-21,2.75,2.75,2.75,2.75,2.75,2.75,No Happy Hour,No Happy Hour,0.0,0.0,Happy Hour 12-7,40,2.75,2.75,2.75,0.61,NaN


In [ ]:
# # fix the happy hour 12-6 recorded wrongly as 12-7 when the "Happy_Hour" is equal "Happy Hour 12-6"
# condition1 = daily_prices_final["HH_Org"] == "Happy Hour 12-7"
# condition2 = daily_prices_final["Happy_Hour"] == "Happy Hour 12-6"
# # and "Slot_18_to_19_Org" is equal 0
# condition3 = daily_prices_final["Slot_18_to_19_Org"] == 0
# # and "D_Org" is equal "Discount" column
# condition4 = daily_prices_final["D_Org"] == daily_prices_final["Discount"]
# mask = condition1 & condition2 & condition3 & condition4
# print(f"Number of rows matching all conditions for Case 1: {mask.sum()}")
# # then we will replace the value of "Happy_Hour_Item" with the value of "Happy_Hour" column
# daily_prices_final.loc[mask, "Happy_Hour_Item"] = daily_prices_final.loc[mask, "Happy_Hour"]
# # replace the value of "Slot_18_to_19" with the value of "Base_Price_Hours"
# daily_prices_final.loc[mask, "Slot_18_to_19"] = daily_prices_final.loc[mask, "Base_Price_Hours"]
# daily_prices_final

In [ ]:
# # if "HH_Org" is equal "Unknown" and "Happy_Hour" is equal  "Happy Hour 12-6" or "Happy Hour 12-7"
# condition1 = daily_prices_final["HH_Org"] == "Unknown"
# condition2 = daily_prices_final["Happy_Hour"].isin(["Happy Hour 12-7", "Happy Hour 12-6"])
# # if "Base_Price_Hours_Org" is equal "Slot_12_to_18_Org" and "Slot_18_to_19_Org" is equal( 0 or previous "Slot_18_to_19_Org")
# condition3 = daily_prices_final["Base_Price_Hours_Org"] == daily_prices_final["Slot_12_to_18_Org"]
# previous_slot_18_to_19 = daily_prices_final["Slot_18_to_19_Org"].shift(1)
# condition4 = (daily_prices_final["Slot_18_to_19_Org"] == 0) | (daily_prices_final["Slot_18_to_19_Org"] == previous_slot_18_to_19)
# # and "Slot_12_to_18" is equal previous "Slot_12_to_18" value
# previous_slot_12_to_18 = daily_prices_final["Slot_12_to_18"].shift(1)
# condition5 = daily_prices_final["Slot_12_to_18"] == previous_slot_12_to_18
# mask = condition1 & condition2 & condition3 & condition4 & condition5
# print(f"Number of rows matching all conditions for Case 2: {mask.sum()}")
# # this means that the happy hour is recorded wrongly as "Unknown"
# # so we will replace the value of "Happy_Hour_Item" with the value of "Happy_Hour" column
# daily_prices_final.loc[mask, "Happy_Hour_Item"] = daily_prices_final.loc[mask, "Happy_Hour"]
# daily_prices_final.loc[mask, "Discount_Item"] = daily_prices_final.loc[mask, "Discount"]
# daily_prices_final.loc[mask, "Slot_18_to_19"] = daily_prices_final.loc[mask, "Base_Price_Hours"]
# # update the base price hours 
# # if "Happy_Hour" is "Happy Hour 12-6"
# # update the base price hours to be equal slot_18_to_19
# # if "Happy_Hour" is "Happy Hour 12-7"
# # update the base price hours to be equal previous base price hours
# daily_prices_final.loc[mask, "Base_Price_Hours"]  = np.where(
#     daily_prices_final.loc[mask, "Happy_Hour"] == "Happy Hour 12-6",
#     daily_prices_final.loc[mask, "Slot_18_to_19"],
#     daily_prices_final.loc[mask, "Base_Price_Hours"].shift(1)
# )
# daily_prices_final

In [ ]:
# UNKNOWN_LABEL = "Unknown"
# VALID_HH_LABELS = ["Happy Hour 12-7", "Happy Hour 12-6"]
# HH_12_6 = "Happy Hour 12-6"

# def correct_unknown_hh_records(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Identifies and corrects records where Happy Hour was mislabeled as 'Unknown'
#     based on a complex set of rules, using a fully vectorized approach.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: The DataFrame with corrections applied.
#     """
    
#     prev_slot_18 = df["Slot_18_to_19"].shift(1)
#     prev_slot_12 = df["Slot_12_to_18"].shift(1)

#     mask = (
#         (df["HH_Org"] == UNKNOWN_LABEL) &
#         df["Happy_Hour"].isin(VALID_HH_LABELS) &
#         (df["Base_Price_Hours_Org"] == df["Slot_12_to_18_Org"]) &
#         ((df["Slot_18_to_19_Org"] == 0) | (df["Slot_18_to_19_Org"] == prev_slot_18)) &
#         (df["Slot_12_to_18"] == prev_slot_12)
#     )
    
#     rows_affected = mask.sum()
#     if rows_affected == 0:
#         print("No rows matched the criteria for Case 2. No changes made.")
#         return df

#     print(f"Number of rows matching all conditions for Case 2: {rows_affected}")



#     df_updated = df.assign(
#         Happy_Hour_Item = np.where(mask, np.nan, df["Happy_Hour_Item"]),
#         Discount_Item   = np.where(mask, np.nan, df["Discount_Item"]),
#         Slot_18_to_19   = np.where(mask, np.nan, df["Slot_18_to_19"]),
#         Base_Price_Hours= np.where(mask, np.nan, df["Base_Price_Hours"]),
#     )

#     return df_updated.ffill().bfill()
# daily_prices_final = correct_unknown_hh_records(daily_prices_final)

In [ ]:
# def correct_unknown_hh_records_case_2(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Identifies and corrects records where Happy Hour was mislabeled as 'Unknown'
#     based on a complex set of rules, using a fully vectorized approach.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: The DataFrame with corrections applied.
#     """
    
#     mask = (
#         (df["HH_Org"] == "Unknown") &
#         (df["Happy_Hour"]== "Happy Hour 12-7") &
#         (df["Base_Price_Hours_Org"] == 0) &
#         (df["Slot_12_to_18_Org"] == 0) &
#         ((df["Slot_18_to_19_Org"] != 0) 
#         ))
    
#     rows_affected = mask.sum()
#     if rows_affected == 0:
#         print("No rows matched the criteria for Case 2. No changes made.")
#         return df

#     print(f"Number of rows matching all conditions for Case 2: {rows_affected}")



#     df_updated = df.assign(
#         Happy_Hour_Item = np.where(mask, df["Happy_Hour"], df["Happy_Hour_Item"]),
#         Discount_Item   = np.where(mask, df["Discount"], df["Discount_Item"]),
#         Slot_12_to_18   = np.where(mask, df["Slot_18_to_19_Org"], df["Slot_12_to_18"]),
#     )

#     return df_updated

# daily_prices_final = correct_unknown_hh_records_case_2(daily_prices_final)
# daily_prices_final

In [ ]:
# def correct_unknown_hh_records_case_3(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Identifies and corrects records where Happy Hour was mislabeled as 'Unknown'
#     based on a complex set of rules, using a fully vectorized approach.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: The DataFrame with corrections applied.
#     """
    
#     mask = (
#         (df["HH_Org"] == "Unknown") &
#         (df["Happy_Hour"]== "Happy Hour 12-7") &
#         (df["Base_Price_Hours_Org"] != 0) &
#         (df["Slot_12_to_18_Org"] != 0) &
#         ((df["Slot_18_to_19_Org"] == 0) 
#         ))
    
#     rows_affected = mask.sum()
#     if rows_affected == 0:
#         print("No rows matched the criteria for Case 2. No changes made.")
#         return df

#     print(f"Number of rows matching all conditions for Case 2: {rows_affected}")



#     df_updated = df.assign(
#         Happy_Hour_Item = np.where(mask, df["Happy_Hour"], df["Happy_Hour_Item"]),
#         Discount_Item   = np.where(mask, df["Discount"], df["Discount_Item"]),
#         Slot_18_to_19   = np.where(mask, df["Slot_12_to_18_Org"], df["Slot_18_to_19"]),
#     )

#     return df_updated

# daily_prices_final = correct_unknown_hh_records_case_3(daily_prices_final)
# daily_prices_final

In [ ]:
# def correct_unknown_hh_records_case_4(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Identifies and corrects records where Happy Hour was mislabeled as 'Unknown'
#     based on a complex set of rules, using a fully vectorized approach.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: The DataFrame with corrections applied.
#     """
    
#     mask = (
#         (df["HH_Org"] == "Unknown") &
#         (df["Happy_Hour"]== "Happy Hour 12-7") &
#         (df["Base_Price_Hours_Org"] == 0) &
#         (df["Slot_12_to_18_Org"] != 0) &
#         ((df["Slot_18_to_19_Org"] == 0) 
#         ))
    
#     rows_affected = mask.sum()
#     if rows_affected == 0:
#         print("No rows matched the criteria for Case 2. No changes made.")
#         return df

#     print(f"Number of rows matching all conditions for Case 2: {rows_affected}")



#     df_updated = df.assign(
#         Happy_Hour_Item = np.where(mask, df["Happy_Hour"], df["Happy_Hour_Item"]),
#         Discount_Item   = np.where(mask, df["Discount"], df["Discount_Item"]),
#         Slot_18_to_19   = np.where(mask, df["Slot_12_to_18_Org"], df["Slot_18_to_19"]),
#     )

#     return df_updated

# daily_prices_final = correct_unknown_hh_records_case_4(daily_prices_final)
# daily_prices_final

In [ ]:
# def correct_unknown_hh_records_case_5(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Identifies and corrects records where Happy Hour was mislabeled as 'Unknown'
#     based on a complex set of rules, using a fully vectorized approach.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: The DataFrame with corrections applied.
#     """
    
#     mask = (
#         (df["HH_Org"] == "Unknown") &
#         (df["Happy_Hour"]== "Happy Hour 12-7") &
#         (df["Base_Price_Hours_Org"] != 0) &
#         (df["Slot_12_to_18_Org"] == 0) &
#         (df["Slot_18_to_19_Org"] != 0) &
#         (df["Base_Price_Hours_Org"] != df["Slot_18_to_19_Org"])
#         )
    
#     rows_affected = mask.sum()
#     if rows_affected == 0:
#         print("No rows matched the criteria for Case 2. No changes made.")
#         return df

#     print(f"Number of rows matching all conditions for Case 2: {rows_affected}")



#     df_updated = df.assign(
#         Happy_Hour_Item = np.where(mask, df["Happy_Hour"], df["Happy_Hour_Item"]),
#         Discount_Item   = np.where(mask, df["Discount"], df["Discount_Item"]),
#         Slot_12_to_18   = np.where(mask, df["Slot_18_to_19_Org"], df["Slot_12_to_18"]),
#     )

#     return df_updated

# daily_prices_final = correct_unknown_hh_records_case_5(daily_prices_final)
# daily_prices_final

In [ ]:
# def correct_hh_records_case_6(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Identifies and corrects records where Happy Hour was mislabeled as 'Unknown'
#     based on a complex set of rules, using a fully vectorized approach.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: The DataFrame with corrections applied.
#     """
    
#     mask = (
#         (df["HH_Org"] == "Happy Hour 12-7") &
#         (df["Happy_Hour"]== "Happy Hour 12-6") 
#         )
    
#     rows_affected = mask.sum()
#     if rows_affected == 0:
#         print("No rows matched the criteria for Case 2. No changes made.")
#         return df

#     print(f"Number of rows matching all conditions for Case 2: {rows_affected}")



#     df_updated = df.assign(
#         Happy_Hour_Item = np.where(mask, df["Happy_Hour"], df["Happy_Hour_Item"]),
#         Discount_Item   = np.where(mask, df["Discount"], df["Discount_Item"]),
#         Slot_18_to_19   = np.where(mask, df["Base_Price_Hours"], df["Slot_18_to_19"]),
#     )

#     return df_updated

# daily_prices_final = correct_hh_records_case_6(daily_prices_final)
# daily_prices_final

In [ ]:
# def correct_unknown_hh_records_case_7(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Identifies and corrects records where Happy Hour was mislabeled as 'Unknown'
#     based on a complex set of rules, using a fully vectorized approach.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: The DataFrame with corrections applied.
#     """
    
#     mask = (
#         (df["HH_Org"] == "Unknown") &
#         (df["Happy_Hour"]== "Happy Hour 12-6") &
#         (df["Base_Price_Hours_Org"] != 0) &
#         (df["Slot_12_to_18_Org"] == 0) &
#         (df["Slot_18_to_19_Org"] == 0) &
#         (df["Base_Price_Hours_Org"] != df["Base_Price_Mode"]) &
#         (df["Base_Price_Hours_Org"] == df["Base_Price_Mode_Next"]) 

#         )
    
#     rows_affected = mask.sum()
#     if rows_affected == 0:
#         print("No rows matched the criteria for Case 2. No changes made.")
#         return df

#     print(f"Number of rows matching all conditions for Case 2: {rows_affected}")



#     df_updated = df.assign(
#         Happy_Hour_Item = np.where(mask, df["Happy_Hour"], df["Happy_Hour_Item"]),
#         Discount_Item   = np.where(mask, df["Discount"], df["Discount_Item"]),
#         Base_Price_Hours   = np.where(mask, df["Base_Price_Mode_Next"], df["Base_Price_Hours"]),
#         Slot_18_to_19  = np.where(mask, df["Base_Price_Mode_Next"], df["Slot_18_to_19"]),
#         Slot_12_to_18  = np.where(mask, df["Base_Price_Mode_Next"]*(1-(df["Discount"]/100)).round(1), df["Slot_12_to_18"]),
#     )

#     return df_updated

# daily_prices_final = correct_unknown_hh_records_case_7(daily_prices_final)
# daily_prices_final

In [ ]:
# def correct_unknown_hh_records_case_8(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Identifies and corrects records where Happy Hour was mislabeled as 'Unknown'
#     based on a complex set of rules, using a fully vectorized approach.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: The DataFrame with corrections applied.
#     """
    
#     mask = (
#         (df["HH_Org"] == "Unknown") &
#         (df["Happy_Hour"]== "Happy Hour 12-6") &
#         (df["Base_Price_Hours_Org"] != 0) &
#         (df["Slot_12_to_18_Org"] == 0) &
#         (df["Slot_18_to_19_Org"] == 0) &
#         (df["Base_Price_Hours_Org"] == df["Base_Price_Mode"]) &
#         (df["Base_Price_Hours_Org"] == df["Base_Price_Mode_Next"]) 

#         )
    
#     rows_affected = mask.sum()
#     if rows_affected == 0:
#         print("No rows matched the criteria for Case 2. No changes made.")
#         return df

#     print(f"Number of rows matching all conditions for Case 2: {rows_affected}")



#     df_updated = df.assign(
#         Happy_Hour_Item = np.where(mask, df["Happy_Hour"], df["Happy_Hour_Item"]),
#         Discount_Item   = np.where(mask, df["Discount"], df["Discount_Item"]),
#         Base_Price_Hours   = np.where(mask, df["Base_Price_Mode"], df["Base_Price_Hours"]),
#         Slot_18_to_19  = np.where(mask, df["Base_Price_Mode"], df["Slot_18_to_19"]),
#         Slot_12_to_18  = np.where(mask, df["Base_Price_Mode"]*(1-(df["Discount"]/100)).round(1), df["Slot_12_to_18"]),
#     )

#     return df_updated

# daily_prices_final = correct_unknown_hh_records_case_8(daily_prices_final)
# daily_prices_final

In [ ]:
# def correct_unknown_hh_records_case_9(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Identifies and corrects records where Happy Hour was mislabeled as 'Unknown'
#     based on a complex set of rules, using a fully vectorized approach.

#     Args:
#         df (pd.DataFrame): The input DataFrame.

#     Returns:
#         pd.DataFrame: The DataFrame with corrections applied.
#     """
    
#     mask = (
#         (df["HH_Org"] == "Unknown") &
#         (df["Happy_Hour"]== "Happy Hour 12-6") &
#         (df["Base_Price_Hours_Org"] != 0) &
#         (df["Slot_18_to_19_Org"] == 0) &
#         (df["Slot_12_to_18_Org"] == df["Base_Price_Hours_Org"]*(1-(df["Discount"]/100)).round(1)) &
#         (df["Base_Price_Hours_Org"] == df["Base_Price_Mode"]) &
#         (df["Base_Price_Hours_Org"] == df["Base_Price_Mode_Next"]) 

#         )
    
#     rows_affected = mask.sum()
#     if rows_affected == 0:
#         print("No rows matched the criteria for Case 2. No changes made.")
#         return df

#     print(f"Number of rows matching all conditions for Case 2: {rows_affected}")



#     df_updated = df.assign(
#         Happy_Hour_Item = np.where(mask, df["Happy_Hour"], df["Happy_Hour_Item"]),
#         Discount_Item   = np.where(mask, df["Discount"], df["Discount_Item"]),
#         Slot_18_to_19  = np.where(mask, df["Base_Price_Hours_Org"], df["Slot_18_to_19"]),
        
#     )

#     return df_updated

# daily_prices_final = correct_unknown_hh_records_case_9(daily_prices_final)
# daily_prices_final